In [1]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def d_sigmoid(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh(x):
    return np.tanh(x)

def d_tanh(x):
    return 1 - np.tanh(x)**2

In [2]:
class LSTM_From_Scratch:
    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size
        np.random.seed(42)
        
        self.Wf = np.random.uniform(-1, 1, (hidden_size, hidden_size + input_size))
        self.bf = np.random.uniform(-1, 1, (hidden_size, 1))
        
        self.Wi = np.random.uniform(-1, 1, (hidden_size, hidden_size + input_size))
        self.bi = np.random.uniform(-1, 1, (hidden_size, 1))
        
        self.Wc = np.random.uniform(-1, 1, (hidden_size, hidden_size + input_size))
        self.bc = np.random.uniform(-1, 1, (hidden_size, 1))
        
        self.Wo = np.random.uniform(-1, 1, (hidden_size, hidden_size + input_size))
        self.bo = np.random.uniform(-1, 1, (hidden_size, 1))
        
        self.Wy = np.random.uniform(-1, 1, (output_size, hidden_size))
        self.by = np.random.uniform(-1, 1, (output_size, 1))

    def forward(self, inputs):
        self.inputs = inputs
        self.h_states = { -1: np.zeros((self.hidden_size, 1)) }
        self.C_states = { -1: np.zeros((self.hidden_size, 1)) }
        self.caches = {}
        
        h_prev = self.h_states[-1]
        C_prev = self.C_states[-1]
        
        for t, x_t in enumerate(inputs):
            x_t = np.array([[x_t]])
            concat_input = np.vstack((h_prev, x_t))
            
            f_t = sigmoid(np.dot(self.Wf, concat_input) + self.bf)
            i_t = sigmoid(np.dot(self.Wi, concat_input) + self.bi)
            C_tilde = tanh(np.dot(self.Wc, concat_input) + self.bc)
            C_t = f_t * C_prev + i_t * C_tilde
            
            o_t = sigmoid(np.dot(self.Wo, concat_input) + self.bo)
            h_t = o_t * tanh(C_t)
            
            self.h_states[t] = h_t
            self.C_states[t] = C_t
            self.caches[t] = (concat_input, f_t, i_t, C_tilde, C_prev, C_t, o_t)
            
            h_prev = h_t
            C_prev = C_t
            
        y_pred = np.dot(self.Wy, self.h_states[len(inputs)-1]) + self.by
        return y_pred

    def backward(self, y_true, y_pred, learning_rate=0.01):
        dy = 2 * (y_pred - y_true)
        dWy = np.dot(dy, self.h_states[len(self.inputs)-1].T)
        dby = dy
        
        dh_next = np.dot(self.Wy.T, dy)
        dC_next = np.zeros_like(self.C_states[0])
        
        dWf, dWi, dWc, dWo = 0, 0, 0, 0
        dbf, dbi, dbc, dbo = 0, 0, 0, 0
        
        for t in reversed(range(len(self.inputs))):
            concat_input, f_t, i_t, C_tilde, C_prev, C_t, o_t = self.caches[t]
            
            do = dh_next * tanh(C_t) * d_sigmoid(np.dot(self.Wo, concat_input) + self.bo)
            dC = dh_next * o_t * d_tanh(C_t) + dC_next
            
            df = dC * C_prev * d_sigmoid(np.dot(self.Wf, concat_input) + self.bf)
            di = dC * C_tilde * d_sigmoid(np.dot(self.Wi, concat_input) + self.bi)
            dC_tilde = dC * i_t * d_tanh(np.dot(self.Wc, concat_input) + self.bc)
            
            dWf += np.dot(df, concat_input.T)
            dbf += df
            dWi += np.dot(di, concat_input.T)
            dbi += di
            dWc += np.dot(dC_tilde, concat_input.T)
            dbc += dC_tilde
            dWo += np.dot(do, concat_input.T)
            dbo += do
            
            dZ = np.dot(self.Wf.T, df) + np.dot(self.Wi.T, di) + np.dot(self.Wc.T, dC_tilde) + np.dot(self.Wo.T, do)
            dh_next = dZ[:self.hidden_size, :]
            dC_next = dC * f_t
            
        self.Wf -= learning_rate * dWf
        self.bf -= learning_rate * dbf
        self.Wi -= learning_rate * dWi
        self.bi -= learning_rate * dbi
        self.Wc -= learning_rate * dWc
        self.bc -= learning_rate * dbc
        self.Wo -= learning_rate * dWo
        self.bo -= learning_rate * dbo
        self.Wy -= learning_rate * dWy
        self.by -= learning_rate * dby

In [3]:
X = [1, 2, 3]
Y = np.array([[4]])

lstm = LSTM_From_Scratch(input_size=1, hidden_size=4, output_size=1)

epochs = 2000
learning_rate = 0.05

for epoch in range(epochs):
    prediction = lstm.forward(X)
    loss = np.mean((prediction - Y)**2)
    lstm.backward(Y, prediction, learning_rate)
    
    if epoch % 200 == 0:
        print(f"Epoch {epoch} | Prediction: {prediction[0][0]:.4f} | Loss: {loss:.4f}")

print(f"\nFinal Output! Expected: 4 | Model predicted: {lstm.forward(X)[0][0]:.4f}")

Epoch 0 | Prediction: -1.7203 | Loss: 32.7224
Epoch 200 | Prediction: 4.0000 | Loss: 0.0000
Epoch 400 | Prediction: 4.0000 | Loss: 0.0000
Epoch 600 | Prediction: 4.0000 | Loss: 0.0000
Epoch 800 | Prediction: 4.0000 | Loss: 0.0000
Epoch 1000 | Prediction: 4.0000 | Loss: 0.0000
Epoch 1200 | Prediction: 4.0000 | Loss: 0.0000
Epoch 1400 | Prediction: 4.0000 | Loss: 0.0000
Epoch 1600 | Prediction: 4.0000 | Loss: 0.0000
Epoch 1800 | Prediction: 4.0000 | Loss: 0.0000

Final Output! Expected: 4 | Model predicted: 4.0000
